# Public-source note

Final H3K27ac signal-selectivity refinement for the four RNA-defined TSS groups. Produces the `RNA_cluster_1.bed` ... `RNA_cluster_4.bed` files used by the downstream FiNeMo/TSS integration. Outputs were stripped and paths sanitized.

In [ ]:
import os, shlex, subprocess, sys
import pandas as pd

import os, shlex, subprocess, sys
import pandas as pd
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap


In [ ]:
#Plot_H3K27ac_cdts_on_TSS_1kb
H3K27ac_filter_by_H3K4me3_out_dir="/path/to/CEBPB_seq/output/bulk_rna/Cluster_all_starting_cell_types/Starting_cell_types_k-means_clusters_k4_tss_1kb_beds/intersectBed_H3K27ac/filter_by_H3K4me3/"

H3K27ac_cdt_dir="/path/to/CEBPB_seq/output/bulk_rna/Cluster_all_starting_cell_types/Starting_cell_types_k-means_clusters_k4_tss_1kb_beds/intersectBed_H3K27ac/filter_by_H3K4me3//H3K27ac_cdts/"
os.makedirs(H3K27ac_cdt_dir, exist_ok=True)

K_clusters =4



In [ ]:
file1_path= '/path/to/seqdata/EL_EC_RNA_ATAC_ChIP_CNR/figure1/Chromhmm/MEF_FBS_Cell2017_H3K27ac_mm10_sorted_dedup.bam'
file2_path= '/path/to/seqdata/EL_EC_RNA_ATAC_ChIP_CNR/figure1/Chromhmm/NSC_H3K27AC_mm10_sorted_dedup.bam'
file3_path=  '/path/to/seqdata/EL_EC_RNA_ATAC_ChIP_CNR/figure1/Chromhmm/ESC_H3K27ac_mm10_sorted_dedup.bam'
file4_path= '/path/to/seqdata/EL_EC_RNA_ATAC_ChIP_CNR/figure1/Chromhmm/BMpreB_H3K27Ac_ChIP_rep1_mm10_sorted_dedup.bam'

bam_dict={    'MEF':file1_path,
       'NSC':file2_path,
    'ESC':file3_path,
        'Bcell':file4_path

    
}

In [ ]:
profile_plot_scr_path="/path/to/binhouse/fpkm_around_bam_regions.py"

In [ ]:
Cluster_label_dict={    'MEF':'RNA_cluster_1',
       'NSC':'RNA_cluster_2',
    'ESC':'RNA_cluster_3',
        'Bcell':'RNA_cluster_4'

    
}
len(Cluster_label_dict.keys()),len(Cluster_label_dict.values())

In [ ]:
samples=['MEF', 'NSC', 'ESC', 'Bcell']
samples

In [ ]:
Cluster_label_dict.values()

In [ ]:
clusters=['RNA_cluster_1', 'RNA_cluster_2', 'RNA_cluster_3', 'RNA_cluster_4']
clusters

In [ ]:
import subprocess
from concurrent.futures import ProcessPoolExecutor, as_completed
import os
import pandas as pd
from pathlib import Path

window = 2000



def run_profile_plot(state, sample_name, bam_path, bed_path, profile_plot_scr_path, window):
    """Run one command for a given state and sample."""

    output_prefix = os.path.join(
       H3K27ac_cdt_dir, f"{state}_{sample_name}"
    )
    cmd = (
        f"python {profile_plot_scr_path} "
        f"-a {bam_path} "
        f"-b {bed_path} "
        f"--window-size {window} "
        f"-o {output_prefix}"
    )

    print(f"[INFO] Running: {cmd}")
    try:
        result = subprocess.run(cmd, shell=True, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        return (state, sample_name, "Success", result.stdout.decode("utf-8"))
    except subprocess.CalledProcessError as e:
        return (state, sample_name, "Error", e.stderr.decode("utf-8"))

# --- Parallel execution ---
tasks = []
with ProcessPoolExecutor(max_workers=44) as executor:  # adjust number of workers based on CPU cores
    for sample, cluster in Cluster_label_dict.items():
        print(sample, cluster)
        # bed_path=H3K27ac_filter_by_other_H3K27ac_out_dir+"/"+cluster+"_intersectBed_"+sample+"_"+"H3K4me3"+".bed"
        bed_path=H3K27ac_filter_by_H3K4me3_out_dir+"/"+cluster+"_intersectBed_"+sample+"_"+"H3K27ac"+".bed"
        print(bed_path)
        for sample_name, bam_path in bam_dict.items():
            print(sample_name)
            tasks.append(executor.submit(
                    run_profile_plot, cluster, sample_name, bam_path, bed_path, profile_plot_scr_path, window
                ))
    for future in as_completed(tasks):
        cluster, sample_name, status, output = future.result()
        print(f"[{status}] {cluster} | {sample_name}")
        if status == "Error":
            print("stderr:\n", output[:500])  # print first few lines only


In [ ]:
for sample, cluster in Cluster_label_dict.items():
    print(sample, cluster)
    dfs = []
    output_cdt_path = os.path.join(
       H3K27ac_cdt_dir,"merged_cdts", f"{cluster}.cdt")
    
    for sample_name in bam_dict.keys():
        input_cdt_path = os.path.join(     H3K27ac_cdt_dir, f"{cluster}_{sample_name}.cdt")
        cols = pd.read_csv(input_cdt_path, sep="\t", nrows=0, comment="#").columns.tolist()
        cdt_df = pd.read_csv(
                input_cdt_path,
                sep="\t",
                usecols=cols     )
        dfs.append(cdt_df)
    combined = pd.concat(dfs, axis=1)
    window_cols = [col for col in combined.columns if col.startswith("WINDOW")]
    combined['sum']=combined[window_cols[4:]].sum(axis=1)
    # combined['sum']=combined[cols[4:]].sum(axis=1)
    combined=combined.sort_values(by="sum", ascending=False)
    del combined['sum']
    combined.to_csv(output_cdt_path, index=None,
                 sep='\t')
    print("Saved: ", output_cdt_path)
    # break;

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from itertools import cycle



# --- gather all sample names to keep colors consistent across subplots ---
# if you already have `sample_names`, we'll use that order; otherwise infer later
unique_samples = list(samples)  # or infer later if needed

# make a simple name fix helper
def normalize_name(sn):
    return 'MEF_D2_OSKM_Cebpb' if sn == 'MEF_D2_SOKM_Cebpb' else sn

# build a color map for samples (consistent across all panels)
color_cycle = cycle(plt.rcParams['axes.prop_cycle'].by_key()['color'])
sample_to_color = {normalize_name(sn): next(color_cycle) for sn in unique_samples}

# plotting helper that draws into an axis (no plt.figure inside)
def plot_metaplot_on_ax(ax, plot_df, group, sample_to_color, collect_legend=False):
    # x = column labels from 2nd to penultimate
    x = plot_df.columns[1:-1]

    handles = []
    labels = []
    # each row is one sample profile
    for _, row in plot_df.iterrows():
        sn = normalize_name(row['sample_name'])
        h, = ax.plot(x, row[1:-1].values, label=sn, linewidth=2,
                     color=sample_to_color.get(sn))
        if collect_legend:
            handles.append(h); labels.append(sn)

    ax.set_ylabel(group)
    ax.grid(True, linewidth=0.4, alpha=0.4)
    return handles, labels

# ---------------- main figure ----------------
# fig, axes = plt.subplots(nrows=7, ncols=1, sharex=True, figsize=(4, 16), constrained_layout=True)
# fig, axes = plt.subplots(nrows=7, ncols=1, sharex=True, figsize=(3, 10), constrained_layout=True)
# fig, axes = plt.subplots(nrows=7, ncols=1, sharex=True, figsize=(2, 8), constrained_layout=True)

fig, axes = plt.subplots(nrows=4, ncols=1, sharex=True, figsize=(2.5, 10), constrained_layout=True)
all_handles, all_labels = [], []

# for sample, cluster in Cluster_label_dict.items():
for ax, cluster in zip(axes, clusters):
    print( cluster)
    dfs = []
    for sample_name in bam_dict.keys():
        input_metaplot_path = os.path.join(     H3K27ac_cdt_dir, f"{cluster}_{sample_name}.metaplot")
        metaplot_df = pd.read_csv(input_metaplot_path, sep='\t', header=None)
        # rename columns so x-columns are strings (like you used before)
        # assumes: first col = something like position index, last col will be sample_name we add
        metaplot_df['sample_name'] = normalize_name(sample_name)
        # give columns numeric indices except last one, to mirror your original indexing
        metaplot_df.columns = list(range(metaplot_df.shape[1]-1)) + ['sample_name']
        dfs.append(metaplot_df)

    combined_metaplot_df = pd.concat(dfs, axis=0, ignore_index=True)

    # only collect legend entries on the first panel to avoid duplicates
    collect = (cluster == clusters[0])
    print(collect)
    handles, labels = plot_metaplot_on_ax(ax, combined_metaplot_df, cluster, sample_to_color, collect_legend=collect)
    if collect:
        # de-duplicate while preserving order
        seen = set()
        for h, lab in zip(handles, labels):
            if lab not in seen:
                all_handles.append(h); all_labels.append(lab)
                seen.add(lab)

axes[-1].set_xlabel('Position')
fig.suptitle('Metaplots for 7 Groups', y=1.02, fontsize=14)

# one common legend at the top
fig.legend(all_handles, all_labels, loc='upper center',
           ncol=min(1, len(all_labels)), frameon=False, bbox_to_anchor=(0.5, 1.2))

plt.show()


## Clean cdts by_Chip

In [ ]:

filter_by_chip_dir="/path/to/CEBPB_seq/output/bulk_rna/Cluster_all_starting_cell_types/Starting_cell_types_k-means_clusters_k4_tss_1kb_beds/intersectBed_H3K27ac/filter_by_H3K4me3/H3K27ac_cdts/filtered_by_H3K27ac_signal/"


In [ ]:
clusters

In [ ]:
H3K27ac_cdt_dir

In [ ]:
cluster="RNA_cluster_1"
print( cluster)
dfs = []
output_cdt_path=filter_by_chip_dir+"/"+cluster+'.cdt'

sample_index=1
for sample_name in bam_dict.keys():
    input_metaplot_path = os.path.join(     H3K27ac_cdt_dir, f"{cluster}_{sample_name}.cdt")
    cols = pd.read_csv(input_metaplot_path, sep="\t", nrows=0, comment="#").columns.tolist()
    cdt_df = pd.read_csv(
            input_metaplot_path,
            sep="\t",
            usecols=cols     )
    cdt_df['sum']=cdt_df[cols[4:]].sum(axis=1)
    print(sample_index)
    cdt_df = cdt_df.add_suffix(f"_{str(sample_index)}")
    sample_index=sample_index+1
    # break;
    dfs.append(cdt_df)
combined = pd.concat(dfs, axis=1)
print("Input: ",cluster, len(combined))
sub_combine=combined[((combined['sum_1']>combined['sum_2']) )]
sub_combine=sub_combine[((sub_combine['sum_1']>sub_combine['sum_3']) )]
sub_combine=sub_combine[((sub_combine['sum_1']>sub_combine['sum_4']) )]
sub_combine=sub_combine.sort_values(by="sum_1", ascending=False)

del sub_combine['sum_1']
del sub_combine['sum_2']
del sub_combine['sum_3']
del sub_combine['sum_4']

print("Output: ",cluster, len(sub_combine))
sub_combine.to_csv(output_cdt_path, index=None,
         sep='\t')
print("Saved: ", output_cdt_path)

In [ ]:
cluster="RNA_cluster_2"
print( cluster)
dfs = []
output_cdt_path=filter_by_chip_dir+"/"+cluster+'.cdt'

sample_index=1
for sample_name in bam_dict.keys():
    input_metaplot_path = os.path.join(     H3K27ac_cdt_dir, f"{cluster}_{sample_name}.cdt")
    cols = pd.read_csv(input_metaplot_path, sep="\t", nrows=0, comment="#").columns.tolist()
    cdt_df = pd.read_csv(
            input_metaplot_path,
            sep="\t",
            usecols=cols     )
    cdt_df['sum']=cdt_df[cols[4:]].sum(axis=1)
    print(sample_index)
    cdt_df = cdt_df.add_suffix(f"_{str(sample_index)}")
    sample_index=sample_index+1
    # break;
    dfs.append(cdt_df)
combined = pd.concat(dfs, axis=1)
print("Input: ",cluster, len(combined))
sub_combine=combined[((combined['sum_2']>combined['sum_1']) )]
sub_combine=sub_combine[((sub_combine['sum_2']>sub_combine['sum_3']) )]
sub_combine=sub_combine[((sub_combine['sum_2']>sub_combine['sum_4']) )]
sub_combine=sub_combine.sort_values(by="sum_2", ascending=False)

del sub_combine['sum_1']
del sub_combine['sum_2']
del sub_combine['sum_3']
del sub_combine['sum_4']


print("Output: ",cluster, len(sub_combine))
sub_combine.to_csv(output_cdt_path, index=None,
         sep='\t')
print("Saved: ", output_cdt_path)

In [ ]:
cluster="RNA_cluster_3"
print( cluster)
dfs = []
output_cdt_path=filter_by_chip_dir+"/"+cluster+'.cdt'

sample_index=1
for sample_name in bam_dict.keys():
    input_metaplot_path = os.path.join(     H3K27ac_cdt_dir, f"{cluster}_{sample_name}.cdt")
    cols = pd.read_csv(input_metaplot_path, sep="\t", nrows=0, comment="#").columns.tolist()
    cdt_df = pd.read_csv(
            input_metaplot_path,
            sep="\t",
            usecols=cols     )
    cdt_df['sum']=cdt_df[cols[4:]].sum(axis=1)
    print(sample_index)
    cdt_df = cdt_df.add_suffix(f"_{str(sample_index)}")
    sample_index=sample_index+1
    # break;
    dfs.append(cdt_df)
combined = pd.concat(dfs, axis=1)
print("Input: ",cluster, len(combined))
sub_combine=combined[((combined['sum_3']>combined['sum_1']) )]
sub_combine=sub_combine[((sub_combine['sum_3']>sub_combine['sum_2']) )]
sub_combine=sub_combine[((sub_combine['sum_3']>sub_combine['sum_4']) )]
sub_combine=sub_combine.sort_values(by="sum_3", ascending=False)

del sub_combine['sum_1']
del sub_combine['sum_2']
del sub_combine['sum_3']
del sub_combine['sum_4']


print("Output: ",cluster, len(sub_combine))
sub_combine.to_csv(output_cdt_path, index=None,
         sep='\t')
print("Saved: ", output_cdt_path)

In [ ]:
cluster="RNA_cluster_4"
print( cluster)
dfs = []
output_cdt_path=filter_by_chip_dir+"/"+cluster+'.cdt'

sample_index=1
for sample_name in bam_dict.keys():
    input_metaplot_path = os.path.join(     H3K27ac_cdt_dir, f"{cluster}_{sample_name}.cdt")
    cols = pd.read_csv(input_metaplot_path, sep="\t", nrows=0, comment="#").columns.tolist()
    cdt_df = pd.read_csv(
            input_metaplot_path,
            sep="\t",
            usecols=cols     )
    cdt_df['sum']=cdt_df[cols[4:]].sum(axis=1)
    print(sample_index)
    cdt_df = cdt_df.add_suffix(f"_{str(sample_index)}")
    sample_index=sample_index+1
    # break;
    dfs.append(cdt_df)
combined = pd.concat(dfs, axis=1)
print("Input: ",cluster, len(combined))
sub_combine=combined[((combined['sum_4']>combined['sum_1']) )]
sub_combine=sub_combine[((sub_combine['sum_4']>sub_combine['sum_2']) )]
sub_combine=sub_combine[((sub_combine['sum_4']>sub_combine['sum_3']) )]
sub_combine=sub_combine.sort_values(by="sum_4", ascending=False)

del sub_combine['sum_1']
del sub_combine['sum_2']
del sub_combine['sum_3']
del sub_combine['sum_4']

print("Output: ",cluster, len(sub_combine))
sub_combine.to_csv(output_cdt_path, index=None,
         sep='\t')
print("Saved: ", output_cdt_path)

## Create new beds based on cdts Cleaned by_cdts

In [ ]:

filtered_bed_dir="/path/to/CEBPB_seq/output/bulk_rna/Cluster_all_starting_cell_types/Starting_cell_types_k-means_clusters_k4_tss_1kb_beds/intersectBed_H3K27ac/filter_by_H3K4me3/H3K27ac_cdts/filtered_by_H3K27ac_signal/RNA_groups_beds/"



In [ ]:
for sample, cluster in Cluster_label_dict.items():
    print(sample, cluster)
    input_cdt_path=filter_by_chip_dir+'/'+cluster+'.cdt'    
    cdt_df = pd.read_csv(
            input_cdt_path,
            sep="\t"  )
    cdt_df[['chrom', 'positions']] = cdt_df['NAME_1'].str.split(':', expand=True)
    cdt_df[['start', 'end']] = cdt_df['positions'].str.split('-', expand=True)
    cdt_df[['chrom', 'start', 'end']].to_csv(filtered_bed_dir+"/"+cluster+'.bed',
                                             sep='\t', header=False, index=False)
    # break;

In [ ]:
for sample, cluster in Cluster_label_dict.items():
    print(sample, cluster)
    filtered_bed_df=pd.read_csv(filtered_bed_dir+"/"+cluster+'.bed',
                                             sep='\t', header=None)
    un_filtered_bed_path=H3K27ac_filter_by_H3K4me3_out_dir+"/"+cluster+"_intersectBed_"+sample+"_"+"H3K27ac"+".bed"
    un_filtered_bed_df=pd.read_csv(un_filtered_bed_path,
                                             sep='\t', header=None)
    print(len(un_filtered_bed_df),len(filtered_bed_df))
    filtered_bed_df=filtered_bed_df.merge(un_filtered_bed_df[[0,1,2,8]], on=[0,1,2])
    print(len(un_filtered_bed_df),len(filtered_bed_df))
    filtered_bed_df.to_csv(filtered_bed_dir+"/"+cluster+'.bed',
                                             sep='\t', header=False, index=False)

In [ ]:
un_filtered_bed_df.head()

In [ ]:
filtered_bed_df.head()

## re-genrate cdts from filtered_beds

In [ ]:

regenrated_cdts_dir="/path/to/CEBPB_seq/output/bulk_rna/Cluster_all_starting_cell_types/Starting_cell_types_k-means_clusters_k4_tss_1kb_beds/intersectBed_H3K27ac/filter_by_H3K4me3/H3K27ac_cdts/filtered_by_H3K27ac_signal/RNA_groups_beds/regenrated_cdts_from_filtered_beds/"


In [ ]:
import subprocess
from concurrent.futures import ProcessPoolExecutor, as_completed
import os
import pandas as pd
from pathlib import Path

window = 2000



def run_profile_plot(state, sample_name, bam_path, bed_path, profile_plot_scr_path, window):
    """Run one command for a given state and sample."""

    output_prefix = os.path.join(
       regenrated_cdts_dir, f"{state}_{sample_name}"
    )
    cmd = (
        f"python {profile_plot_scr_path} "
        f"-a {bam_path} "
        f"-b {bed_path} "
        f"--window-size {window} "
        f"-o {output_prefix}"
    )

    print(f"[INFO] Running: {cmd}")
    try:
        result = subprocess.run(cmd, shell=True, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        return (state, sample_name, "Success", result.stdout.decode("utf-8"))
    except subprocess.CalledProcessError as e:
        return (state, sample_name, "Error", e.stderr.decode("utf-8"))

# --- Parallel execution ---
tasks = []
with ProcessPoolExecutor(max_workers=44) as executor:  # adjust number of workers based on CPU cores
    for sample, cluster in Cluster_label_dict.items():
        print(sample, cluster)
        # bed_path=H3K27ac_filter_by_other_H3K27ac_out_dir+"/"+cluster+"_intersectBed_"+sample+"_"+"H3K4me3"+".bed"
        bed_path=filtered_bed_dir+"/"+cluster+".bed"
        print(bed_path)
        for sample_name, bam_path in bam_dict.items():
            print(sample_name)
            tasks.append(executor.submit(
                    run_profile_plot, cluster, sample_name, bam_path, bed_path, profile_plot_scr_path, window
                ))
    for future in as_completed(tasks):
        cluster, sample_name, status, output = future.result()
        print(f"[{status}] {cluster} | {sample_name}")
        if status == "Error":
            print("stderr:\n", output[:500])  # print first few lines only


In [ ]:
for sample, cluster in Cluster_label_dict.items():
    print(sample, cluster)
    dfs = []
    output_cdt_path = os.path.join(
       regenrated_cdts_dir,"merged_cdts", f"{cluster}.cdt")
    
    for sample_name in bam_dict.keys():
        input_cdt_path = os.path.join(     regenrated_cdts_dir, f"{cluster}_{sample_name}.cdt")
        cols = pd.read_csv(input_cdt_path, sep="\t", nrows=0, comment="#").columns.tolist()
        cdt_df = pd.read_csv(
                input_cdt_path,
                sep="\t",
                usecols=cols     )
        dfs.append(cdt_df)
    combined = pd.concat(dfs, axis=1)
    window_cols = [col for col in combined.columns if col.startswith("WINDOW")]
    combined['sum']=combined[window_cols[4:]].sum(axis=1)
    # combined['sum']=combined[cols[4:]].sum(axis=1)
    combined=combined.sort_values(by="sum", ascending=False)
    del combined['sum']
    combined.to_csv(output_cdt_path, index=None,
                 sep='\t')
    print("Saved: ", output_cdt_path)
    # break;

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from itertools import cycle



# --- gather all sample names to keep colors consistent across subplots ---
# if you already have `sample_names`, we'll use that order; otherwise infer later
unique_samples = list(samples)  # or infer later if needed

# make a simple name fix helper
def normalize_name(sn):
    return 'MEF_D2_OSKM_Cebpb' if sn == 'MEF_D2_SOKM_Cebpb' else sn

# build a color map for samples (consistent across all panels)
color_cycle = cycle(plt.rcParams['axes.prop_cycle'].by_key()['color'])
sample_to_color = {normalize_name(sn): next(color_cycle) for sn in unique_samples}

# plotting helper that draws into an axis (no plt.figure inside)
def plot_metaplot_on_ax(ax, plot_df, group, sample_to_color, collect_legend=False):
    # x = column labels from 2nd to penultimate
    x = plot_df.columns[1:-1]

    handles = []
    labels = []
    # each row is one sample profile
    for _, row in plot_df.iterrows():
        sn = normalize_name(row['sample_name'])
        h, = ax.plot(x, row[1:-1].values, label=sn, linewidth=2,
                     color=sample_to_color.get(sn))
        if collect_legend:
            handles.append(h); labels.append(sn)

    ax.set_ylabel(group)
    ax.grid(True, linewidth=0.4, alpha=0.4)
    return handles, labels

# ---------------- main figure ----------------
# fig, axes = plt.subplots(nrows=7, ncols=1, sharex=True, figsize=(4, 16), constrained_layout=True)
# fig, axes = plt.subplots(nrows=7, ncols=1, sharex=True, figsize=(3, 10), constrained_layout=True)
# fig, axes = plt.subplots(nrows=7, ncols=1, sharex=True, figsize=(2, 8), constrained_layout=True)

fig, axes = plt.subplots(nrows=4, ncols=1, sharex=True, figsize=(2.5, 9), constrained_layout=True)
all_handles, all_labels = [], []

# for sample, cluster in Cluster_label_dict.items():
for ax, cluster in zip(axes, clusters):
    print( cluster)
    dfs = []
    for sample_name in bam_dict.keys():
        input_metaplot_path = os.path.join(     regenrated_cdts_dir, f"{cluster}_{sample_name}.metaplot")
        metaplot_df = pd.read_csv(input_metaplot_path, sep='\t', header=None)
        # rename columns so x-columns are strings (like you used before)
        # assumes: first col = something like position index, last col will be sample_name we add
        metaplot_df['sample_name'] = normalize_name(sample_name)
        # give columns numeric indices except last one, to mirror your original indexing
        metaplot_df.columns = list(range(metaplot_df.shape[1]-1)) + ['sample_name']
        dfs.append(metaplot_df)

    combined_metaplot_df = pd.concat(dfs, axis=0, ignore_index=True)

    # only collect legend entries on the first panel to avoid duplicates
    collect = (cluster == clusters[0])
    print(collect)
    handles, labels = plot_metaplot_on_ax(ax, combined_metaplot_df, cluster, sample_to_color, collect_legend=collect)
    if collect:
        # de-duplicate while preserving order
        seen = set()
        for h, lab in zip(handles, labels):
            if lab not in seen:
                all_handles.append(h); all_labels.append(lab)
                seen.add(lab)

axes[-1].set_xlabel('Position')
fig.suptitle('H3K27ac', y=1.02, fontsize=14)

# one common legend at the top
fig.legend(all_handles, all_labels, loc='upper center',
           ncol=min(2, len(all_labels)), frameon=False, bbox_to_anchor=(0.5, 1.1))

plt.show()
